In [1]:
%pip install transformers accelerate torch sentencepiece bitsandbytes aiohttp requests --quiet together nltk rouge-score bert-score scikit-learn pandas tqdm

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.6 MB/s eta 0:00:00


In [ ]:
from dotenv import load_dotenv
import os, json, time
from together import Together
import numpy as np

load_dotenv()
api_key = "fillin"

client = Together(api_key=api_key)

MODEL = "chrisperez04_345b/upstage/SOLAR-10.7B-Instruct-v1.0-fce547f6"

INPUT_FILE = "/content/data/query_data.jsonl"
RESULTS_DIR = "/content/results/"

# Ensure results directory exists
os.makedirs(RESULTS_DIR, exist_ok=True)

# Define the range of repetition penalties to test
REPETITION_PENALTIES = np.arange(1.0, 1.9, 0.2).tolist()  # [1.0, 1.1, 1.2, ..., 1.8]
print(f"Testing repetition penalties: {[round(p, 1) for p in REPETITION_PENALTIES]}")

Testing repetition penalties: [1.0, 1.2, 1.4, 1.6, 1.8]


In [3]:
# Base generation config (will be updated with each repetition penalty)
BASE_GEN_CFG = dict(
    max_tokens=512,
    temperature=0.2,
    top_p=0.9,
    top_k=60,
    num_beams=1,
    stop=["</s>"],
)

In [4]:
def run_query(prompt: str, gen_cfg: dict):
    """
    Send one prompt to the model (chat or completions endpoint) and
    return (text, latency).  Works for Together dedicated endpoints.
    """
    import time
    start = time.time()

    # --- First try chat.completions (some endpoints accept this) ---
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            **gen_cfg,
        )
        text = resp.choices[0].message.content if resp.choices else ""
        return text.strip(), time.time() - start
    except Exception as e1:
        err1 = str(e1)
        print(f"Error in chat completion: {err1}")
        return "", 0.0

In [5]:
def run_inference_for_penalty(repetition_penalty: float):
    """
    Run inference for all queries with a specific repetition penalty.
    Returns the output file path.
    """
    penalty_str = f"{repetition_penalty:.1f}".replace(".", "_")
    output_file = os.path.join(RESULTS_DIR, f"Llama-2-7b-chat-hf_penalty_{penalty_str}.jsonl")

    # Update generation config with current penalty
    gen_cfg = BASE_GEN_CFG.copy()
    gen_cfg["repetition_penalty"] = repetition_penalty

    print(f"\n{'='*80}")
    print(f"Running inference with repetition_penalty = {repetition_penalty:.1f}")
    print(f"Output file: {output_file}")
    print(f"{'='*80}\n")

    # Load input queries
    queries = []
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    queries.append(json.loads(line))
                except json.JSONDecodeError:
                    print(f"⚠️ Skipping malformed line: {line[:60]}...")

    # Run queries and save results
    with open(output_file, "w", encoding="utf-8") as out:
        for i, query_obj in enumerate(queries, start=1):
            prompt = query_obj.get("prompt", "")
            qid = query_obj.get("id", i)

            print(f"[{i}] id={qid} — sending prompt of length {len(prompt)} chars")

            try:
                output_text, latency = run_query(prompt, gen_cfg)
                print(f"✔ Done in {latency:.2f}s, {len(output_text)} chars output.")

                result = {
                    "id": qid,
                    "prompt": prompt,
                    "output": output_text,
                    "latency": latency,
                    "repetition_penalty": repetition_penalty,
                }

                # Copy over additional fields if present
                for key in ["context", "anchor"]:
                    if key in query_obj:
                        result[key] = query_obj[key]

                out.write(json.dumps(result, ensure_ascii=False) + "\n")
                out.flush()

            except Exception as e:
                print(f"✘ Error: {e}")
                result = {
                    "id": qid,
                    "prompt": prompt,
                    "output": "",
                    "error": str(e),
                    "repetition_penalty": repetition_penalty,
                }
                out.write(json.dumps(result, ensure_ascii=False) + "\n")
                out.flush()

    print(f"\n✅ Completed inference for penalty {repetition_penalty:.1f}")
    print(f"Results saved to: {output_file}\n")
    return output_file

In [6]:
def evaluate(results_path: str, out_csv: str = None):
    """
    Evaluate model outputs using various metrics.
    Returns a dictionary of mean scores.
    """
    import pandas as pd
    from tqdm import tqdm
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    from rouge_score import rouge_scorer
    from sklearn.metrics import f1_score
    from bert_score import score as bert_score

    if out_csv is None:
        out_csv = os.path.splitext(results_path)[0] + ".csv"

    # Load results (expecting "context" and "output")
    rows = []
    with open(results_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue  # skip blank lines
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                print(f"⚠️ Skipping malformed line: {line[:60]}...")
                continue

    # Filter to rows that have both fields (graceful on errors)
    eval_rows = [r for r in rows if isinstance(r.get("context"), str) and isinstance(r.get("output"), str)]

    if not eval_rows:
        print("No rows with both 'context' and 'output' found; skipping evaluation.")
        return None

    ids = [r.get("id") for r in eval_rows]
    anchors = [r.get("anchor", "") for r in eval_rows]
    refs = [r["context"] for r in eval_rows]
    hyps = [r["output"]  for r in eval_rows]

    # Metric helpers
    rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    smoothie = SmoothingFunction().method1

    def rouge_l(h, r):
        return rouge.score(r, h)["rougeL"].fmeasure

    def bleu(h, r):
        h_toks = h.split()
        r_toks = [r.split()]
        return sentence_bleu(r_toks, h_toks, smoothing_function=smoothie)

    def token_f1(h, r):
        h_toks = h.split()
        r_toks = r.split()
        vocab = list(set(h_toks + r_toks))
        h_vec = [1 if t in h_toks else 0 for t in vocab]
        r_vec = [1 if t in r_toks else 0 for t in vocab]
        try:
            return f1_score(r_vec, h_vec)
        except ValueError:
            return 0.0

    # Compute classical metrics
    rouge_vals, bleu_vals, f1_vals = [], [], []
    for h, r in tqdm(list(zip(hyps, refs)), total=len(hyps), desc="Classical metrics"):
        rouge_vals.append(rouge_l(h, r))
        bleu_vals.append(bleu(h, r))
        f1_vals.append(token_f1(h, r))

    # BERTScore (vectorized)
    print("Computing BERTScore...")
    P, R, F = bert_score(hyps, refs, lang="en", verbose=False)
    bert_f1_vals = F.numpy().tolist()

    # 95% CI via normal approx
    def ci95(x):
        x = np.asarray(x, dtype=float)
        mean = x.mean()
        se = x.std(ddof=1) / max(1, np.sqrt(len(x)))
        low, high = mean - 1.96 * se, mean + 1.96 * se
        return mean, low, high

    # Per-row CSV
    df = pd.DataFrame({
        "id": ids,
        "anchor": anchors,
        "ROUGE-L": rouge_vals,
        "BLEU": bleu_vals,
        "Token-F1": f1_vals,
        "BERTScore": bert_f1_vals,
    })
    df.to_csv(out_csv, index=False)
    print(f"Saved detailed scores → {out_csv}")

    # Summary
    r_m, r_l, r_h = ci95(rouge_vals)
    b_m, b_l, b_h = ci95(bleu_vals)
    f_m, f_l, f_h = ci95(f1_vals)
    bs_m, bs_l, bs_h = ci95(bert_f1_vals)

    print("\n=== Mean ± 95% CI ===")
    print(f"ROUGE-L   : {r_m:.4f}  (95% CI {r_l:.4f}–{r_h:.4f})")
    print(f"BLEU      : {b_m:.4f}  (95% CI {b_l:.4f}–{b_h:.4f})")
    print(f"Token-F1  : {f_m:.4f}  (95% CI {f_l:.4f}–{f_h:.4f})")
    print(f"BERTScore : {bs_m:.4f} (95% CI {bs_l:.4f}–{bs_h:.4f})")

    return {
        "ROUGE-L": r_m,
        "BLEU": b_m,
        "Token-F1": f_m,
        "BERTScore": bs_m,
    }

In [7]:
# Main execution: Run inference and evaluation for each repetition penalty
import pandas as pd

all_results = []

for penalty in REPETITION_PENALTIES:
    penalty = round(penalty, 1)  # Ensure clean float values

    # Run inference
    output_file = run_inference_for_penalty(penalty)

    # Run evaluation
    print(f"\n{'='*80}")
    print(f"Evaluating results for repetition_penalty = {penalty:.1f}")
    print(f"{'='*80}\n")

    penalty_str = f"{penalty:.1f}".replace(".", "_")
    csv_file = os.path.join(RESULTS_DIR, f"Llama-2-7b-chat-hf_penalty_{penalty_str}.csv")

    scores = evaluate(output_file, out_csv=csv_file)

    if scores:
        scores["repetition_penalty"] = penalty
        all_results.append(scores)

    print("\n" + "="*80 + "\n")

# Create summary comparison table
if all_results:
    summary_df = pd.DataFrame(all_results)
    summary_df = summary_df[['repetition_penalty', 'ROUGE-L', 'BLEU', 'Token-F1', 'BERTScore']]

    summary_file = os.path.join(RESULTS_DIR, "summary_comparison.csv")
    summary_df.to_csv(summary_file, index=False)

    print("\n" + "="*100)
    print("FINAL SUMMARY: COMPARISON ACROSS ALL REPETITION PENALTIES")
    print("="*100 + "\n")
    print(summary_df.to_string(index=False))
    print(f"\nSummary saved to: {summary_file}")

    # Find best penalty for each metric
    print("\n" + "="*100)
    print("BEST REPETITION PENALTY FOR EACH METRIC:")
    print("="*100 + "\n")
    for metric in ['ROUGE-L', 'BLEU', 'Token-F1', 'BERTScore']:
        best_idx = summary_df[metric].idxmax()
        best_penalty = summary_df.loc[best_idx, 'repetition_penalty']
        best_score = summary_df.loc[best_idx, metric]
        print(f"{metric:12s}: {best_score:.4f} at repetition_penalty = {best_penalty:.1f}")
else:
    print("\n⚠️ No evaluation results were generated.")


Running inference with repetition_penalty = 1.0
Output file: /content/results/Llama-2-7b-chat-hf_penalty_1_0.jsonl

[1] id=1 — sending prompt of length 1137 chars
✔ Done in 7.19s, 959 chars output.
[2] id=2 — sending prompt of length 1695 chars
✔ Done in 5.43s, 1459 chars output.
[3] id=3 — sending prompt of length 1335 chars
✔ Done in 5.93s, 1129 chars output.
[4] id=4 — sending prompt of length 1491 chars
✔ Done in 2.49s, 341 chars output.
[5] id=5 — sending prompt of length 1546 chars
✔ Done in 5.28s, 1332 chars output.
[6] id=6 — sending prompt of length 1514 chars
✔ Done in 5.40s, 1234 chars output.
[7] id=7 — sending prompt of length 1411 chars
✔ Done in 5.97s, 1258 chars output.
[8] id=8 — sending prompt of length 1349 chars
✔ Done in 9.03s, 1726 chars output.
[9] id=9 — sending prompt of length 1356 chars
✔ Done in 5.71s, 1262 chars output.
[10] id=10 — sending prompt of length 1468 chars
✔ Done in 3.26s, 775 chars output.
[11] id=11 — sending prompt of length 1279 chars
✔ Don

Classical metrics: 100%|██████████| 230/230 [00:03<00:00, 61.64it/s]


Computing BERTScore...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

Saved detailed scores → /content/results/Llama-2-7b-chat-hf_penalty_1_6.csv

=== Mean ± 95% CI ===
ROUGE-L   : 0.0786  (95% CI 0.0698–0.0873)
BLEU      : 0.0030  (95% CI 0.0021–0.0040)
Token-F1  : 0.0406  (95% CI 0.0364–0.0448)
BERTScore : 0.7828 (95% CI 0.7792–0.7863)



Running inference with repetition_penalty = 1.8
Output file: /content/results/Llama-2-7b-chat-hf_penalty_1_8.jsonl

[1] id=1 — sending prompt of length 1137 chars
✔ Done in 6.45s, 787 chars output.
[2] id=2 — sending prompt of length 1695 chars
✔ Done in 4.72s, 875 chars output.
[3] id=3 — sending prompt of length 1335 chars
✔ Done in 4.76s, 1122 chars output.
[4] id=4 — sending prompt of length 1491 chars
✔ Done in 4.49s, 1005 chars output.
[5] id=5 — sending prompt of length 1546 chars
✔ Done in 8.40s, 2957 chars output.
[6] id=6 — sending prompt of length 1514 chars
✔ Done in 8.61s, 3008 chars output.
[7] id=7 — sending prompt of length 1411 chars
✔ Done in 8.85s, 2926 chars output.
[8] id=8 — sending prompt of len

Classical metrics: 100%|██████████| 230/230 [00:03<00:00, 60.16it/s]


Computing BERTScore...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Saved detailed scores → /content/results/Llama-2-7b-chat-hf_penalty_1_8.csv

=== Mean ± 95% CI ===
ROUGE-L   : 0.0591  (95% CI 0.0522–0.0659)
BLEU      : 0.0008  (95% CI 0.0006–0.0009)
Token-F1  : 0.0155  (95% CI 0.0138–0.0173)
BERTScore : 0.7655 (95% CI 0.7619–0.7692)



FINAL SUMMARY: COMPARISON ACROSS ALL REPETITION PENALTIES

 repetition_penalty  ROUGE-L     BLEU  Token-F1  BERTScore
                1.0 0.852006 0.768208  0.864266   0.960419
                1.2 0.715877 0.580502  0.700749   0.937508
                1.4 0.268150 0.097075  0.234170   0.843015
                1.6 0.078594 0.003017  0.040618   0.782797
                1.8 0.059057 0.000769  0.015546   0.765549

Summary saved to: /content/results/summary_comparison.csv

BEST REPETITION PENALTY FOR EACH METRIC:

ROUGE-L     : 0.8520 at repetition_penalty = 1.0
BLEU        : 0.7682 at repetition_penalty = 1.0
Token-F1    : 0.8643 at repetition_penalty = 1.0
BERTScore   : 0.9604 at repetition_penalty = 1.0
